In [ ]:
import pandas as pd

In [ ]:
!pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 51.5 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.


In [ ]:
import wfdb
import os

dataset_path = '/content/drive/MyDrive/sleep_apnea/apnea-ecg-database-1.0.0'

record_names = [
    # # Group A: 20 records
    # "a01", "a02", "a03", "a04", "a05", "a06", "a07", "a08", "a09", "a10",
    # "a11", "a12", "a13", "a14", "a15", "a16", "a17", "a18", "a19", "a20"
    # # Group B: 5 records
    # "b01", "b02", "b03", "b04", "b05",
    # # Group C: 10 records
    "c01", "c02", "c03", "c04", "c05", "c06", "c07", "c08", "c09", "c10",
    # "x01", "x02", "x03", "x04", "x05", "x06", "x07", "x08", "x09", "x10",
    # "x11", "x12", "x13", "x14", "x15", "x16", "x17", "x18", "x19", "x20",
    # "x21", "x22", "x23", "x24", "x25", "x26", "x27", "x28", "x29", "x30",
    # "x31", "x32", "x33", "x34", "x35"
]





In [ ]:
segmented_data = []

for record_name in record_names:
    try:
        # Load the signal and annotations
        record_path = f"{dataset_path}/{record_name}"
        record = wfdb.rdrecord(record_path)
        annotation = wfdb.rdann(record_path, 'apn')

        raw_signal = record.p_signal[:, 0]  # ECG signal (first channel)
        sampling_frequency = record.fs
        labels = annotation.symbol  # Apnea/Normal labels

        # Apply scaling using gain and baseline
        gain = record.adc_gain[0]  # Gain for first channel
        baseline = record.baseline[0]  # Baseline for first channel
        scaled_signal = (raw_signal - baseline) / gain

        # Calculate segment length
        segment_length = int(sampling_frequency * 60)  # 1-minute segments

        # Truncate the signal to match the labels
        total_samples = len(labels) * segment_length
        truncated_signal = scaled_signal[:total_samples]

        # Segment and label
        for i, label in enumerate(labels):
            start = i * segment_length
            end = (i + 1) * segment_length
            segment = truncated_signal[start:end]

            segmented_data.append({
                "Record_ID": record_name,
                "Segment_ID": i + 1,
                "Signal_Data": segment.tolist(),
                "Label": label
            })

        print(f"Processed record: {record_name} (Segments: {len(labels)})")

    except Exception as e:
        print(f"Error processing record {record_name}: {e}")


Processed record: c01 (Segments: 484)
Processed record: c02 (Segments: 502)
Processed record: c03 (Segments: 454)
Processed record: c04 (Segments: 482)
Processed record: c05 (Segments: 466)
Processed record: c06 (Segments: 468)
Processed record: c07 (Segments: 429)
Processed record: c08 (Segments: 513)
Processed record: c09 (Segments: 468)
Processed record: c10 (Segments: 431)


In [ ]:
for i in range(len(segmented_data)):
  segmented_data[i]['Signal_Data'] = [format(num, '.7f') for num in segmented_data[i]['Signal_Data']]

In [ ]:
pd.DataFrame(segmented_data).to_csv("/content/drive/MyDrive/sleep_apnea/C_DataSet.csv", index=False)